# Sparse3D GPU worker

This notebook turns your Colab session into a GPU worker for the Sparse3D web app.
**You only need to do three things:**

1. **Runtime > Change runtime type > T4 GPU**
2. **Runtime > Run all**
3. Copy the **connection code** printed at the bottom (starts with `S3D1.`) into the app: *Settings > Execution > Connect*

Your photos go only to this Colab session. Stop the notebook (or close the tab) to disconnect.
The first run downloads the model weights (a few GB) and installs COLMAP, so allow about 10 minutes; later runs in the
same session are instant.

### 1. Check the GPU

In [ ]:
import subprocess
out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True)
assert out.returncode == 0 and 'GPU' in out.stdout, (
    'No GPU found. Use Runtime > Change runtime type > T4 GPU, then Run all again.')
print(out.stdout.strip())

### 2. Install a CUDA-enabled COLMAP

Ubuntu's `apt` COLMAP has no CUDA, so dense reconstruction would fail. The conda-forge build has it.

In [ ]:
import os, subprocess, sys
os.makedirs('/content/mm', exist_ok=True)
!cd /content/mm && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj bin/micromamba > /dev/null
!/content/mm/bin/micromamba create -y -q -p /content/colmap-env -c conda-forge colmap
COLMAP = '/content/colmap-env/bin/colmap'
help_text = subprocess.run([COLMAP, 'help'], capture_output=True, text=True).stdout
print(help_text.splitlines()[0])
assert 'with CUDA' in help_text, 'This COLMAP build has no CUDA; build it from source (https://colmap.github.io/install.html).'

### 3. Get the code, the neural model and its weights

In [ ]:
!rm -rf /content/Sparse3D && git clone --depth 1 https://github.com/IshaVats04/Sparse3D /content/Sparse3D
%cd /content/Sparse3D
!pip install -q -r requirements.txt transformers git+https://github.com/facebookresearch/vggt.git

In [ ]:
# Download the weights now so the first experiment does not wait for them.
from huggingface_hub import snapshot_download
for repo in ('facebook/VGGT-1B', 'depth-anything/Depth-Anything-V2-Small-hf'):
    try:
        snapshot_download(repo)
        print('ready:', repo)
    except Exception as exc:
        print('skipped', repo, '->', exc)

### 4. Tunnel client

In [ ]:
!curl -sL -o /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared

### 5. Start the worker and print the connection code

In [ ]:
import base64, json, re, secrets, subprocess, sys, threading, time, urllib.request

TOKEN = secrets.token_urlsafe(24)
env = dict(os.environ, SPARSE3D_API_TOKEN=TOKEN, SPARSE3D_COLMAP=COLMAP,
           PATH='/content/colmap-env/bin:' + os.environ['PATH'], SPARSE3D_DATA_DIR='/content/sparse3d_data')
server = subprocess.Popen([sys.executable, 'src/backend/app.py', '--host', '127.0.0.1', '--port', '8000'], env=env)
tunnel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)

url = None
for line in tunnel.stdout:
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
    if m:
        url = m.group(0)
        break
threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()

for _ in range(60):                       # wait until the worker answers through the tunnel
    try:
        req = urllib.request.Request(url + '/api/health', headers={'X-Sparse3D-Token': TOKEN})
        health = json.load(urllib.request.urlopen(req, timeout=5))['environment']
        break
    except Exception:
        time.sleep(2)
else:
    raise RuntimeError('The worker did not start; run this cell again.')

code = 'S3D1.' + base64.urlsafe_b64encode(json.dumps({'u': url, 't': TOKEN}, separators=(',', ':')).encode()).decode().rstrip('=')
print('GPU:', health.get('gpu_name'), '| COLMAP CUDA:', health.get('colmap_gpu_usable'),
      '| neural backend:', health.get('neural_backend'))
print()
print('=' * 70)
print('CONNECTION CODE (paste into the app: Settings > Execution > Connect)')
print()
print(code)
print()
print('=' * 70)
print('Keep this tab open. Stop this cell to disconnect.')

In [ ]:
# Keeps the session alive while you use the app. Interrupt this cell to stop the worker.
try:
    while True:
        time.sleep(60)
except KeyboardInterrupt:
    server.terminate(); tunnel.terminate()
    print('Worker stopped.')